In [15]:
import os
import requests
import pandas as pd
from tqdm import tqdm

df = pd.read_csv("C:/Users/flori/Downloads/archelect_search.csv",sep=",")
df = df[df["contexte-election"]=="présidentielle"]

C:\Users\flori\AppData\Local\Temp\ipykernel_23048\481703485.py:6: DtypeWarning: Columns (8,9,10,12,28,29,30,31,32,33,34,35,36,37,38,39,40,41) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("C:/Users/flori/Downloads/archelect_search.csv",sep=",")


In [8]:
df_pres

,id,date,subject,title,contexte-election,contexte-tour,cote,departement,departement-nom,departement-insee,...,suppleant-age-calcule,suppleant-age-tranche,suppleant-profession,suppleant-mandat-en-cours,suppleant-mandat-passe,suppleant-associations,suppleant-autres-statuts,suppleant-soutien,suppleant-liste,suppleant-decorations
5016,EL033_P_1965_001b,1965-12-05,BARBU Marcel; France; Elections présidentielle...,Election présidentielle de 1965 : profession d...,présidentielle,1,EL033,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5017,EL033_P_1965_002,1965-12-05,DE GAULLE Charles; France; Elections président...,Election présidentielle de 1965 : profession d...,présidentielle,1,EL033,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5018,EL033_P_1965_003,1965-12-05,LECANUET Jean; France; Elections présidentiell...,Election présidentielle de 1965 : profession d...,présidentielle,1,EL033,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5019,EL033_P_1965_004,1965-12-05,MARCILHACY Pierre; France; Elections président...,Election présidentielle de 1965 : profession d...,présidentielle,1,EL033,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5020,EL033_P_1965_005,1965-12-05,MITTERRAND François; France; Elections préside...,Election présidentielle de 1965 : profession d...,présidentielle,1,EL033,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32915,EL245_P_2012_036,2012-04-22,France; Elections présidentielles; Ve Républiq...,Election présidentielle de 2012 : profession d...,présidentielle,1,EL245,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
32916,EL245_P_2012_037,2012-04-22,France; Elections présidentielles; Ve Républiq...,Election présidentielle de 2012 : profession d...,présidentielle,1,EL245,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
32917,EL245_P_2012_038,2012-04-22,France; Elections présidentielles; Ve Républiq...,Election présidentielle de 2012 : profession d...,présidentielle,1,EL245,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
32918,EL245_P_2012_049,2012-05-06,France; Elections présidentielles; Ve Républiq...,Election présidentielle de 2012 : profession d...,présidentielle,2,EL245,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [16]:
# ======================
# PARAMÈTRES
# ======================
OUTPUT_DIR = "archelec_ocr_texts"
SLEEP_BETWEEN_REQUESTS = 0.3  # très important
TIMEOUT = 10                 # plus court = mieux
MAX_RETRIES = 5

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ======================
# SESSION HTTP ROBUSTE
# ======================
session = requests.Session()

# retries = Retry(
#     total=MAX_RETRIES,
#     backoff_factor=1.5,              # délai croissant
#     status_forcelist=[429, 500, 502, 503, 504],
#     allowed_methods=["GET"]
# )

# adapter = HTTPAdapter(max_retries=retries)
# session.mount("https://", adapter)
# session.mount("http://", adapter)

# ======================
# FICHIERS DÉJÀ PRÉSENTS
# ======================
existing_ids = {
    os.path.splitext(f)[0]
    for f in os.listdir(OUTPUT_DIR)
    if f.endswith(".txt")
}

print(f"📁 Déjà présents : {len(existing_ids)}")

df["id_str"] = df["id"].astype(str)

df_to_download = df[
    (~df["id_str"].isin(existing_ids)) &
    (df["ocr_url"].notna()) &
    (df["ocr_url"].str.strip() != "")
]

📁 Déjà présents : 7267


In [10]:
df_to_download.iloc[0]["ocr_url"]

'https://ia804704.us.archive.org/17/items/EL033_P_1965_006/EL033_P_1965_006_djvu.txt'

In [17]:
import os
import time
import requests
import pandas as pd
from tqdm import tqdm
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry



print(f"⬇️ À télécharger : {len(df_to_download)}")

# ======================
# TÉLÉCHARGEMENT
# ======================
def download_ocr(url, out_path, doc_id=None):
    try:
        r = session.get(url, timeout=TIMEOUT)
        r.raise_for_status()

        # Vérifications utiles
        content_type = r.headers.get("Content-Type", "")
        size = len(r.content)

        if size < 100:
            print(f"⚠️ [{doc_id}] ÉCHEC : contenu trop court ({size} octets)")
            return False

        if "text" not in content_type:
            print(f"⚠️ [{doc_id}] ÉCHEC : Content-Type = {content_type}")
            return False

        with open(out_path, "wb") as f:
            f.write(r.content)

        return True

    except requests.exceptions.Timeout:
        print(f"⏱️ [{doc_id}] TIMEOUT sur {url}")
        return False

    except requests.exceptions.HTTPError as e:
        print(f"🌐 [{doc_id}] HTTP ERROR {r.status_code} sur {url}")
        return False

    except Exception as e:
        print(f"❌ [{doc_id}] ERREUR INATTENDUE : {e}")
        return False

# ======================
# BOUCLE PRINCIPALE
# ======================
failed_ids = []
START_AT = 0

for i, (_, row) in enumerate(
    tqdm(df_to_download.iterrows(), total=len(df_to_download)),
    start=0
):
    if i < START_AT:
        continue

    doc_id = row["id_str"]
    url = row["ocr_url"]
    out_path = os.path.join(OUTPUT_DIR, f"{doc_id}.txt")

    ok = download_ocr(url, out_path)
    if not ok:
        failed_ids.append(doc_id)
        print(f"❌ Échec téléchargement : {doc_id}")
    else:
        print(f"✅ Téléchargé : {doc_id}")

    time.sleep(SLEEP_BETWEEN_REQUESTS)

# ======================
# LOG DES ÉCHECS
# ======================
if failed_ids:
    pd.DataFrame({"id": failed_ids}).to_csv(
        "archelec_failed_downloads.csv", index=False
    )
    print(f"⚠️ Échecs : {len(failed_ids)} (log enregistré)")

print("✅ Téléchargement terminé")


⬇️ À télécharger : 36


  0%|          | 0/36 [00:00<?, ?it/s]

⏱️ [None] TIMEOUT sur https://ia804704.us.archive.org/17/items/EL033_P_1965_006/EL033_P_1965_006_djvu.txt
❌ Échec téléchargement : EL033_P_1965_006


  3%|▎         | 1/36 [00:10<06:02, 10.34s/it]

⏱️ [None] TIMEOUT sur https://ia804701.us.archive.org/26/items/EL033_P_1965_014/EL033_P_1965_014_djvu.txt
❌ Échec téléchargement : EL033_P_1965_014


...

⏱️ [None] TIMEOUT sur https://ia804702.us.archive.org/25/items/EL245_P_2012_036/EL245_P_2012_036_djvu.txt
❌ Échec téléchargement : EL245_P_2012_036


 97%|█████████▋| 35/36 [06:02<00:10, 10.34s/it]

⏱️ [None] TIMEOUT sur https://ia904706.us.archive.org/27/items/EL245_P_2012_050/EL245_P_2012_050_djvu.txt
❌ Échec téléchargement : EL245_P_2012_050


100%|██████████| 36/36 [06:12<00:00, 10.35s/it]

⚠️ Échecs : 36 (log enregistré)
✅ Téléchargement terminé
